## Forward modeling

The propagation of seismic waves in heterogeneous, isotropic, elastic earth media can be expressed by the elastodynamic equations:

\begin{equation}
\left\{\begin{array}{ll} 
\dfrac{\partial \sigma_{xx}}{\partial t} - (\lambda+2\mu)\dfrac{\partial v_{x}}{\partial x}-\lambda\dfrac{\partial v_{z}}{\partial z}=f_{\sigma_{xx}},\\
\dfrac{\partial\sigma_{zz}}{\partial t}-(\lambda+2\mu)\dfrac{\partial v_{z}}{\partial z} -\lambda\dfrac{\partial v_{x}}{\partial x}=f_{\sigma_{zz}},\\
\dfrac{\partial \sigma_{zx}}{\partial t} -\mu\Big(\dfrac{\partial v_{x}}{\partial z} + \dfrac{\partial v_{z}}{\partial x}\Big) = f_{\sigma_{zx}},\\
\rho\dfrac{\partial v_x}{\partial t} - \Big(\dfrac{\partial \sigma_{xx}}{\partial x} + \dfrac{\partial \sigma_{xz}}{\partial z} )=0,\\
\rho\dfrac{\partial v_z}{\partial t} - \Big(\dfrac{\partial \sigma_{zx}}{\partial x}+\dfrac{\partial \sigma_{zz}}{\partial z}\Big)=0 .
\end{array}\right.
\end{equation}


where $\vec{v}=(v_x,~v_z)$ are the horizontal and vertical particle velocity fields, $\sigma=(\sigma_{xx},~\sigma_{zz},~\sigma_{xz}$) are the stress fields, $f=$($f_{\sigma_{xx}},~f_{\sigma_{xx}})$
are the source terms, $\rho$ is density, $\lambda$ and $\mu$ are the Lame parameters. The elastic wave equation given described above can be organized using a compact formulation as presented by Chen and Sacchi (2020):

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-{\bf D} \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-{\bf C D}^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}

being ${\bf C}$ the isotropic elastic tensor in Voigt notation, ${\bf D}$ is a collection of spatial differential operators, defined as

\begin{equation}
\begin{split}
   {\bf C}=\left(\begin{array}{ccc}
   \lambda+2 \mu & \lambda & 0 \\
   \lambda & \lambda+2 \mu & 0 \\ 
   0 & 0 & \mu \end{array}\right)~~\text{e}~~
{\bf D}=\left(\begin{array}{ccc}  
\dfrac{\partial}{\partial x}& 0 &\dfrac{\partial}{\partial z} \\
0 &  \dfrac{\partial}{\partial z}&\dfrac{\partial}{\partial x}
\end{array}\right)  .
\end{split}
\end{equation}

In our compact formulation, the stress tensor $(\sigma)$ is represented is written in vectorial form:

\begin{equation}
\sigma=\left(\begin{array}{c}
 \sigma_{xx}  \\ 
 \sigma_{zz}  \\ 
 \sigma_{zx}
\end{array}\right)
\end{equation}

The conversion from matrix to vector form of the stress tensor is done using the vec() function. An example of this application is $\sigma$=vec($\sigma$).

In [ ]:
from examples.seismic.source import RickerSource, TimeAxis
from examples.seismic import setup_geometry, PointSource, Receiver
from examples.seismic import SeismicModel
from examples.seismic.stiffness.model import ISOSeismicModel
from examples.seismic.stiffness.utils import C_Matrix, D, S, vec
from devito import (Eq, Operator, VectorTimeFunction, TensorTimeFunction,
                    VectorFunction, solve)
from examples.seismic.stiffness import iso_elastic_setup
import numpy as np
import matplotlib.pyplot as plt
from devito import configuration, norm
configuration['log-level'] = 'WARNING'

from examples.seismic.tutorials.lista_utils import *

from ipywidgets import interact, widgets

%matplotlib widget

In [ ]:
nx, nz = 301, 301
dx, dz = 10, 10
shape = (nx, nz)
origin = (0., 0.)
nlayers = 3
nbl = 50
space_order = 8
dtype = np.float32

# Model physical parameters
vp = np.ones(shape, dtype=dtype) * 1.5
vp[:, nx // 3 : ] = 2.5
vp[:, nx // 3 * 2 : ] = 3.5
vs = vp * 0.5
vs[vp == 1.5] = 0
rho = vp
b = 1 / rho

a1, a2, a3, b1, b2, b3 = 5.5, 6.9, 2.2, 3.4, 4.7, 1.8
rhoq, rhoc, rhow, rhoh = 2.65, 2.55, 1., 0.1

clay = (a2 * (vs - b1) + b2 * (a1 - vp)) / (a3 * b2 - a2 * b3)
Phi = (a1 - a3 * clay - vp) / a2
Sw = (((rho - (clay * (rhoc - rhoq) + rhoq) * (1 - Phi)) / Phi) - rhoh) / (rhow - rhoh)

model = Model(vp=vp, vs=vs, b=b, Phi=Phi, clay=clay, Sw=Sw, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs='damp')

model.rho = model._gen_phys_param(rho, 'rho', space_order=space_order)
model.Ip = model._gen_phys_param(vp*rho, 'Ip', space_order=space_order, is_param=True)
model.Is = model._gen_phys_param(vs*rho, 'Is', space_order=space_order, is_param=True)

In [ ]:
plot_model(model)

In [ ]:
f0 = 0.010

# Time step in ms and time range:
t0, tn = 0., 3000.
dt = model.critical_dt * 0.1
ns = 1
ng = model.shape[0]

s_pos = np.zeros((ng,2)) # Posição dos receptores
s_pos[:, 0] = model.domain_size[0] * 0.5 # Posição dos receptores (x)
s_pos[:, 1] = 40 # Posição dos receptores (y)

g_pos = np.zeros((ng,2)) # Posição dos receptores
g_pos[:, 0] = np.linspace(0, model.domain_size[0], ng) # Posição dos receptores (x)

src, rec = get_src_rec(model, t0, tn, dt, f0, ns, ng, s_pos, g_pos)

plot_aquisition_setup(model, src, rec[0])

## Building the forward  operator

The **elastic_forward(...)** function is responsible for forward modeling and makes use of the compact first-order elastic  equation system described below:

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-D \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-C D^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}


In [ ]:
# Forward modeling

param = ['vp-vs-rho', 'Ip-Is-rho', 'lam-mu', 'PCS']

[display(C_Matrix(model, param[i])) for i in range(4)]

V0, sigma0, rec0 = elastic_forward(model, src, rec, param=param[0], save=False)

src, rec = get_src_rec(model, t0, tn, dt, f0, ns, ng, s_pos, g_pos)
V1, sigma1, rec1 = elastic_forward(model, src, rec, param=param[1], save=False)

src, rec = get_src_rec(model, t0, tn, dt, f0, ns, ng, s_pos, g_pos)
V2, sigma2, rec2 = elastic_forward(model, src, rec, param=param[2], save=False)

src, rec = get_src_rec(model, t0, tn, dt, f0, ns, ng, s_pos, g_pos)
V3, sigma3, rec3 = elastic_forward(model, src, rec, param=param[3], save=False)

## Plotting vx, vz and pressure fields shots

In [ ]:
plot_matrix = np.asarray([[rec_ for rec_ in rec] for rec in [rec0, rec1, rec2, rec3]])

aspect_ratio = model.shape[0] / model.shape[1]

plt_options_model = {'cmap': 'Greys', 'extent': [model.origin[0],
                                                 model.origin[0] + model.domain_size[0],
                                                 model.origin[1] + model.domain_size[1],
                                                 model.origin[1]], 'aspect':aspect_ratio}

rows = plot_matrix.shape[0]
cols = plot_matrix.shape[1]
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(5*cols, 5*rows), sharex=True, sharey=True)

for row in range(rows):
    for col in range(cols):
        amax = max(abs(plot_matrix[row, col].data.min()), plot_matrix[row, col].data.max()) / 100

        axes[row,col].imshow(plot_matrix[row,col].data,
                             **plt_options_model,
                             vmin=-amax,
                             vmax=amax
        )
        
axes[0, 0].set_title('$V_{x}$')
axes[0, 1].set_title('$V_{z}$')
axes[0, 2].set_title('$\sigma_{xx} + \sigma_{zz}$')

axes[0, 0].set_ylabel('$V_p, V_s, \\rho$ \n\n twt$(ms)$')
axes[1, 0].set_ylabel('$I_p, I_s, \\rho$ \n\n twt$(ms)$')
axes[2, 0].set_ylabel('$\\lambda, \\mu$ \n\n twt$(ms)$')
axes[3, 0].set_ylabel('$\\Phi$, C, $S_w$ \n\n twt$(ms)$')

[axes[3, i].set_xlabel('$x(m)$') for i in range(3)]

fig.tight_layout()
plt.show()

# fig.savefig('seismograms.png', dpi=300)

## Plotting the wavefields snapshots for the particle velocity vector and the stress tensor components

In [ ]:
plot_matrix = np.asarray([[V0[0], V0[1], sigma0[0], sigma0[1], sigma0[2]],
                              [V1[0], V1[1], sigma1[0], sigma1[1], sigma1[2]],
                              [V2[0], V2[1], sigma2[0], sigma2[1], sigma2[2]],
                              [V3[0], V3[1], sigma3[0], sigma3[1], sigma3[2]]])

aspect_ratio = model.shape[0] / model.shape[1]

plt_options_model = {'cmap': 'Greys', 'extent': [model.origin[0],
                                                 model.origin[0] + model.domain_size[0],
                                                 model.origin[1] + model.domain_size[1],
                                                 model.origin[1]]}

rows = plot_matrix.shape[0]
cols = plot_matrix.shape[1]
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(3*cols, 2.6*rows), sharex=True, sharey=True)

def update(snap):
    for row in range(rows):
        for col in range(cols):
            amax = max(abs(plot_matrix[row, col].data[snap].min()), plot_matrix[row, col].data[snap].max()) / 10

            axes[row,col].clear()
            axes[row,col].imshow(plot_matrix[row,col].data[snap].T,
                                **plt_options_model,
                                vmin=-amax,
                                vmax=amax
            )
            
    axes[0, 0].set_title('$V_{x}$')
    axes[0, 1].set_title('$V_{z}$')
    axes[0, 2].set_title('$\sigma_{xx}$')
    axes[0, 3].set_title('$\sigma_{zz}$')
    axes[0, 4].set_title('$\sigma_{xz}$')

    axes[0, 0].set_ylabel('$V_p, V_s, \\rho$ \n\n z$(m)$')
    axes[1, 0].set_ylabel('$I_p, I_s, \\rho$ \n\n z$(m)$')
    axes[2, 0].set_ylabel('$\\lambda, \\mu$ \n\n z$(m)$')
    axes[3, 0].set_ylabel('$\\Phi$, C, $S_w$ \n\n twt$(ms)$')

    [axes[3, i].set_xlabel('$x(m)$') for i in range(0, 5)]

interact(update, snap=widgets.IntSlider(value=400, min=0, max=src.time_range.num-1))

fig.tight_layout()
plt.show()